# 🫁 Chest X-ray Intelligence Studio — GPU Backend Server on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VEDANTH0517/chest-xray-intelligence-studio/blob/main/colab/Run_Backend_Google_Colab.ipynb)
![GPU](https://img.shields.io/badge/GPU-T4%20%7C%20V100%20%7C%20A100-success.svg)
![Framework](https://img.shields.io/badge/FastAPI-PyTorch%202.0-blue.svg)
![Tunnel](https://img.shields.io/badge/Cloudflare%20Tunnel-Zero%20Config-orange.svg)

This notebook runs the complete **Chest X-ray Intelligence Studio** backend on Google Colab's free GPU (T4 / V100 / A100) and exposes it via a secure **Cloudflare Tunnel**.

### Features:
- ⚡ **GPU-Accelerated Inference**: 4 independent architectures (ViT-B/16, DenseNet121, CapsNet, GAT).
- 🌐 **Instant Public HTTPS URL**: Access the full single-page dashboard directly from your browser on any device.
- 🎯 **Grounded Lesion Localization**: Bounding boxes labeled directly on radiographs.
- 💎 **Optimal Medical Fidelity**: Evaluates radiograph diagnostic enhancement with **PSNR ≥ 40 dB**.
- 📚 **Factually Grounded GraphRAG**: Verified NCBI Bookshelf & StatPearls literature evidence.

In [ ]:
# @title 1. Verify GPU Hardware Acceleration
# @markdown Checks CUDA availability and allocates GPU memory for deep learning inference.
import torch

print("=" * 65)
print("CHEST X-RAY INTELLIGENCE STUDIO -- HARDWARE CHECK")
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"Active GPU      : {device_name} ({vram_gb} GB VRAM)")
    print("Status          : [READY] GPU Acceleration is active for all models.")
else:
    print("Active Device   : CPU (Fallback)")
    print("TIP             : To enable GPU, go to Runtime -> Change runtime type -> Select 'T4 GPU'.")
print("=" * 65)

In [ ]:
# @title 2. Clone Latest Codebase from GitHub
# @markdown Clones or pulls the latest version of Chest X-ray Intelligence Studio.
import os
from pathlib import Path

REPO_URL = "https://github.com/VEDANTH0517/chest-xray-intelligence-studio.git"
REPO_DIR = "/content/chest-xray-intelligence-studio"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists. Pulling latest code...")
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
print(f"\nActive working directory: {os.getcwd()}")

In [ ]:
# @title 3. Install Required Dependencies & Cloudflare Tunnel
# @markdown Installs all Python dependencies and the Cloudflare Tunnel client.
print("Installing requirements.txt and tunnel tools...")
!pip install -q -r requirements.txt
!pip install -q pycloudflared nest-asyncio uvicorn python-multipart

# Install standalone Cloudflare Tunnel binary for Linux
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✓ Dependencies and Cloudflare Tunnel installed successfully!")

In [ ]:
# @title 4. Model Checkpoints & Pretrained Weights
# @markdown Verifies checkpoints in checkpoints/ or sets up model backbones.
from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints = {
    "ViT-B/16": CHECKPOINT_DIR / "vit.pt",
    "DenseNet121": CHECKPOINT_DIR / "densenet.pt",
    "CapsNet": CHECKPOINT_DIR / "capsnet.pt",
    "GAT": CHECKPOINT_DIR / "gat.pt",
}

print("Checking checkpoint readiness:")
missing = []
for name, path in checkpoints.items():
    if path.exists() and path.stat().st_size > 1000:
        print(f"  ✓ {name:15}: {path.name} ({path.stat().st_size / (1024*1024):.1f} MB)")
    else:
        print(f"  ✗ {name:15}: Missing ({path.name})")
        missing.append(name)

if missing:
    print("\n" + "=" * 65)
    print("OPTIONAL: If you have trained checkpoints in Google Drive:")
    print("1. Uncomment and run:")
    print("   from google.colab import drive; drive.mount('/content/drive')")
    print("   !cp /content/drive/MyDrive/checkpoints/*.pt checkpoints/")
    print("2. Or train them with notebooks/CheXpert_Four_Models_OpenSet_Training.ipynb")
    print("=" * 65)
else:
    print("\n✓ All model weights verified!")

In [ ]:
# @title 5. Launch GPU Backend Server & Start Public Cloudflare Tunnel
# @markdown Starts the Uvicorn/FastAPI backend and creates a secure public HTTPS link.
import time
import subprocess
import re
from google.colab import output

# Terminate previous instances
!pkill -f "uvicorn" > /dev/null 2>&1
!pkill -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

print("Starting FastAPI server on GPU (port 8000)...")
server_proc = subprocess.Popen(
    ["python", "app.py", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Allow server to initialize
time.sleep(3)

print("Opening Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "=" * 70)
    print("🚀 CHEST X-RAY INTELLIGENCE STUDIO IS LIVE ON GPU!")
    print("=" * 70)
    print(f"👉 Studio Dashboard URL : {public_url}")
    print(f"📖 Swagger API Docs    : {public_url}/docs")
    print(f"🩺 Health Status API    : {public_url}/api/health")
    print("=" * 70)
    print("Click the 'Studio Dashboard URL' above to open the application in your browser!")
    print("=" * 70 + "\n")
else:
    print("Tunnel took longer than expected. Opening via Colab port viewer:")
    output.serve_kernel_port_as_window(8000)


In [ ]:
# @title 6. Stream Live GPU Server Logs (Optional)
# @markdown Displays real-time request logs, inference latency, and PSNR calculations.
print("Streaming live server logs (Press the 'Stop' button on this cell to halt output):\n")
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\nLog viewer stopped. The server remains running in the background.")

In [ ]:
# @title 7. Stop Backend Server & Cloudflare Tunnel
# @markdown Run this cell when you are finished using the application.
!pkill -f "uvicorn"
!pkill -f "cloudflared"
print("✓ Backend server and Cloudflare tunnel stopped cleanly.")